# LOB Data Preview and Reconstruction

Preview the raw Bybit BTCUSDT files, then reconstruct **every file** into
one-second order-book rows. Combine all days into one in-memory Polars
DataFrame named `df`, sorted by UTC date and time, and save it as Parquet.

Raw messages contain snapshots and deltas. Prices are USDT per BTC;
quantities are BTC. Daily results are cached so later runs can reuse them.

In [1]:
from pathlib import Path
from IPython.display import display

try:
    import polars as pl
except ModuleNotFoundError:
    %pip install -q "polars>=2.1,<3"
    import polars as pl

pl.Config.set_tbl_rows(10)
pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(36)
pl.Config.set_fmt_table_cell_list_len(2)
print(f"Polars version: {pl.__version__}")

Polars version: 2.1.0


In [2]:
# Works when the kernel starts in the project root or in LOB/.
candidates = [Path.cwd() / "Data", Path.cwd().parent / "Data"]
DATA_DIR = next((path.resolve() for path in candidates if path.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Open this notebook from the HFT project root or LOB folder.")

files = sorted(DATA_DIR.glob("*_BTCUSDT_ob200.data"))
if not files:
    raise FileNotFoundError(f"No BTCUSDT order-book files found in {DATA_DIR}")

total_bytes = sum(path.stat().st_size for path in files)
print(f"Files: {len(files)}")
print(f"Date range: {files[0].name[:10]} to {files[-1].name[:10]}")
print(f"Raw size: {total_bytes / 1_000_000_000:.2f} GB")

Files: 30
Date range: 2026-09-24 to 2026-10-23
Raw size: 33.75 GB


In [3]:
# Explicit types preserve nested price/quantity lists, including empty deltas.
schema = {
    "topic": pl.String,
    "type": pl.String,
    "ts": pl.Int64,
    "data": pl.Struct({
        "s": pl.String,
        "b": pl.List(pl.List(pl.String)),
        "a": pl.List(pl.List(pl.String)),
        "u": pl.Int64,
        "seq": pl.Int64,
    }),
    "cts": pl.Int64,
}

# lob references every file without loading the whole month into RAM.
lob = (
    pl.scan_ndjson(
        [path.as_posix() for path in files],
        schema=schema,
        include_file_paths="source_file",
        low_memory=True,
    )
    .unnest("data")
    .rename({
        "type": "message_type",
        "s": "symbol",
        "b": "bid_updates",
        "a": "ask_updates",
        "u": "update_id",
        "seq": "sequence",
    })
    .with_columns(
        pl.from_epoch("ts", time_unit="ms")
        .dt.replace_time_zone("UTC").alias("timestamp_utc"),
        pl.col("source_file").str.split("/").list.last(),
    )
    .select(
        "source_file", "timestamp_utc", "symbol", "message_type",
        "bid_updates", "ask_updates", "update_id", "sequence",
        "ts", "cts", "topic",
    )
)

print(f"Combined table: all messages from {len(files)} files (lazy).")

Combined table: all messages from 30 files (lazy).


In [4]:
# Only collect the first 10 rows, not the full 34 GB dataset.
N_PREVIEW_ROWS = 10
preview = lob.head(N_PREVIEW_ROWS).collect()
print(f"Preview: {preview.height} rows x {preview.width} columns")
display(preview)

Preview: 10 rows x 11 columns


source_file,timestamp_utc,symbol,message_type,bid_updates,ask_updates,update_id,sequence,ts,cts,topic
str,"datetime[μs, UTC]",str,str,list[list[str]],list[list[str]],i64,i64,i64,i64,str
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.129 UTC,"""BTCUSDT""","""snapshot""","[[""84355.20"", ""0.949""], … [""84321.50"", ""0.009""]]","[[""84355.30"", ""4.536""], … [""84391.40"", ""0.014""]]",53936686,815049327159,1790208000129,1790208000127,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.229 UTC,"""BTCUSDT""","""delta""","[[""84355.20"", ""0.797""], … [""84321.50"", ""0""]]","[[""84355.30"", ""3.697""], … [""84391.40"", ""0""]]",53936687,815049327497,1790208000229,1790208000227,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.329 UTC,"""BTCUSDT""","""delta""","[[""84355.20"", ""1.444""], … [""84322.00"", ""0""]]","[[""84355.30"", ""3.066""], … [""84391.10"", ""0""]]",53936688,815049327836,1790208000329,1790208000324,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.428 UTC,"""BTCUSDT""","""delta""","[[""84351.00"", ""0.005""], … [""84324.10"", ""0""]]","[[""84359.80"", ""0.015""], … [""84390.40"", ""0.003""]]",53936689,815049327985,1790208000428,1790208000423,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.529 UTC,"""BTCUSDT""","""delta""","[[""84353.00"", ""0""], … [""84324.40"", ""1.659""]]","[[""84358.90"", ""0.017""], … [""84390.80"", ""0""]]",53936690,815049328139,1790208000529,1790208000525,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.629 UTC,"""BTCUSDT""","""delta""","[[""84355.20"", ""1.814""], … [""84322.10"", ""0""]]","[[""84368.20"", ""0.023""], … [""84390.70"", ""1.930""]]",53936691,815049328326,1790208000629,1790208000626,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.729 UTC,"""BTCUSDT""","""delta""","[[""84355.20"", ""1.887""], … [""84322.00"", ""0.119""]]","[[""84355.30"", ""3.070""], … [""84390.70"", ""0""]]",53936692,815049328524,1790208000729,1790208000721,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.828 UTC,"""BTCUSDT""","""delta""","[[""84355.20"", ""1.886""], … [""84322.00"", ""0""]]","[[""84379.90"", ""0.016""], … [""84390.70"", ""1.930""]]",53936693,815049328708,1790208000828,1790208000825,"""orderbook.200.BTCUSDT"""
"""2026-09-24_BTCUSDT_ob200.data""",2026-09-24 00:00:00.928 UTC,"""BTCUSDT""","""delta""","[[""84355.20"", ""1.880""], … [""84321.30"", ""0.002""]]","[[""84355.30"", ""3.068""], … [""84390.40"", ""0.001""]]",53936694,815049328933,1790208000928,1790208000926,"""orderbook.200.BTCUSDT"""


In [5]:
# Inspect the nested values without printing hundreds of price levels.
first = preview.row(0, named=True)
print(f"First message type: {first['message_type']}")
print("First 3 bid pairs [price, quantity]:", first["bid_updates"][:3])
print("First 3 ask pairs [price, quantity]:", first["ask_updates"][:3])

First message type: snapshot
First 3 bid pairs [price, quantity]: [['84355.20', '0.949'], ['84355.10', '0.001'], ['84354.10', '0.001']]
First 3 ask pairs [price, quantity]: [['84355.30', '4.536'], ['84355.40', '0.002'], ['84356.20', '0.002']]


## Reconstruct every file and sample every second

A snapshot replaces the previous book. A delta sets the new quantity
at each price; quantity zero removes that price. Each file is read
in order without loading its raw messages into memory.

At each whole-second boundary, use only messages with `ts` at or
before that time, including all equal-timestamp updates. Skip
empty/crossed books and quotes older than two seconds. Restrict each
output to the UTC date in its filename, then concatenate and sort.
Initial seconds before a day's first snapshot can be absent.

In [6]:
import json
import math
from datetime import date


def reconstruct_one_day(path, interval_ms=1000, max_age_ms=2000):
    """Stream one raw file into a compact table of last-known quotes."""
    if interval_ms <= 0 or max_age_ms < 0:
        raise ValueError("interval_ms must be positive; max_age_ms cannot be negative")

    bids, asks = {}, {}
    rows = []
    last_ts = None
    next_sample = None
    stats = {
        "messages": 0, "snapshots": 0, "deltas": 0,
        "stale_samples_skipped": 0, "invalid_samples_skipped": 0,
        "max_message_gap_ms": 0,
    }

    def record_sample(sample_ts):
        age_ms = sample_ts - last_ts
        if age_ms > max_age_ms:
            stats["stale_samples_skipped"] += 1
            return
        if not bids or not asks:
            stats["invalid_samples_skipped"] += 1
            return
        best_bid, best_ask = max(bids), min(asks)
        if best_bid >= best_ask:
            stats["invalid_samples_skipped"] += 1
            return
        rows.append({
            "timestamp_ms": sample_ts,
            "best_bid": best_bid,
            "bid_qty": bids[best_bid],
            "best_ask": best_ask,
            "ask_qty": asks[best_ask],
            "mid_price": round((best_bid + best_ask) / 2, 8),
            "spread": round(best_ask - best_bid, 8),
            "book_age_ms": age_ms,
        })

    # A large buffer avoids slow small reads on Windows-mounted drives.
    with path.open("rb", buffering=1024 * 1024) as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            message = json.loads(line)
            ts = int(message["ts"])
            data = message["data"]
            kind = message["type"]
            if data["s"] != "BTCUSDT":
                raise ValueError(f"Unexpected symbol at line {line_number}")
            if kind not in {"snapshot", "delta"}:
                raise ValueError(f"Unexpected message type at line {line_number}")
            if last_ts is not None and ts < last_ts:
                raise ValueError(f"Timestamp moved backwards at line {line_number}")

            if last_ts is not None:
                stats["max_message_gap_ms"] = max(
                    stats["max_message_gap_ms"], ts - last_ts
                )
                # Emit BEFORE applying this message, so future data cannot leak in.
                # Equality is deferred until all messages at that timestamp arrive.
                while next_sample < ts:
                    record_sample(next_sample)
                    next_sample += interval_ms

            if kind == "snapshot":
                bids.clear()
                asks.clear()
                stats["snapshots"] += 1
            else:
                if stats["snapshots"] == 0:
                    raise ValueError("A snapshot is required before any delta")
                stats["deltas"] += 1

            for side, updates in ((bids, data["b"]), (asks, data["a"])):
                for price_text, quantity_text in updates:
                    price, quantity = float(price_text), float(quantity_text)
                    if not math.isfinite(price) or not math.isfinite(quantity):
                        raise ValueError(f"Non-finite level at line {line_number}")
                    if price <= 0 or quantity < 0:
                        raise ValueError(f"Invalid price/quantity at line {line_number}")
                    if quantity == 0:
                        side.pop(price, None)
                    else:
                        side[price] = quantity

            last_ts = ts
            if next_sample is None:
                next_sample = ((ts + interval_ms - 1) // interval_ms) * interval_ms
            stats["messages"] += 1

    # Include the final boundary only if it was actually observed.
    while last_ts is not None and next_sample <= last_ts:
        record_sample(next_sample)
        next_sample += interval_ms

    table = pl.DataFrame(rows, schema={
        "timestamp_ms": pl.Int64,
        "best_bid": pl.Float64, "bid_qty": pl.Float64,
        "best_ask": pl.Float64, "ask_qty": pl.Float64,
        "mid_price": pl.Float64, "spread": pl.Float64,
        "book_age_ms": pl.Int64,
    }).with_columns(
        pl.from_epoch("timestamp_ms", time_unit="ms")
        .dt.replace_time_zone("UTC").alias("timestamp_utc")
    ).select(
        "timestamp_utc", "best_bid", "bid_qty", "best_ask", "ask_qty",
        "mid_price", "spread", "book_age_ms",
    )
    # Some archive files end with a snapshot from the following UTC day.
    sample_date = date.fromisoformat(path.name[:10])
    inside_day = table.filter(pl.col("timestamp_utc").dt.date() == sample_date)
    stats["outside_day_samples_dropped"] = table.height - inside_day.height
    table = inside_day
    stats["sample_rows"] = table.height
    return table, stats

In [7]:
import ast
import hashlib
import inspect

INTERVAL_MS = 1000
MAX_BOOK_AGE_MS = 2000
PROCESSED_DIR = DATA_DIR.parent / "LOB" / "processed"
CACHE_DIR = PROCESSED_DIR / "daily"
OUTPUT_PATH = PROCESSED_DIR / "btcusdt_lob_1s.parquet"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Invalidate cached days when reconstruction logic changes.
reconstruction_ast = ast.dump(
    ast.parse(inspect.getsource(reconstruct_one_day)), include_attributes=False
)
RECONSTRUCTION_SIGNATURE = hashlib.sha256(reconstruction_ast.encode()).hexdigest()


def load_or_reconstruct_day(path):
    source_stat = path.stat()
    fingerprint = {
        "cache_format": 1,
        "source_path": str(path.resolve()),
        "source_size": source_stat.st_size,
        "source_mtime_ns": source_stat.st_mtime_ns,
        "interval_ms": INTERVAL_MS,
        "max_age_ms": MAX_BOOK_AGE_MS,
        "reconstruction": RECONSTRUCTION_SIGNATURE,
    }
    cache_path = CACHE_DIR / f"{path.stem}_1s.parquet"
    metadata_path = cache_path.with_suffix(".json")
    if cache_path.exists() and metadata_path.exists():
        try:
            metadata = json.loads(metadata_path.read_text())
        except (OSError, json.JSONDecodeError):
            metadata = None
        if isinstance(metadata, dict) and metadata.get("fingerprint") == fingerprint:
            table = pl.read_parquet(cache_path)
            if table.height != metadata["stats"]["sample_rows"]:
                raise ValueError(f"Cached row count mismatch: {cache_path.name}")
            return table, metadata["stats"], True

    table, stats = reconstruct_one_day(
        path, interval_ms=INTERVAL_MS, max_age_ms=MAX_BOOK_AGE_MS
    )
    if table.is_empty():
        raise ValueError(f"No valid one-second rows in {path.name}")

    # Publish complete files atomically, never partially written caches.
    temporary_parquet = cache_path.with_suffix(".parquet.tmp")
    table.write_parquet(temporary_parquet, compression="zstd")
    temporary_parquet.replace(cache_path)
    temporary_metadata = metadata_path.with_suffix(".json.tmp")
    temporary_metadata.write_text(json.dumps({
        "fingerprint": fingerprint, "stats": stats
    }, indent=2) + "\n")
    temporary_metadata.replace(metadata_path)
    return table, stats, False

In [8]:
from time import perf_counter

start = perf_counter()
daily_tables = []
daily_diagnostics = []
for index, path in enumerate(files, start=1):
    daily_table, stats, cached = load_or_reconstruct_day(path)
    daily_tables.append(daily_table)
    daily_diagnostics.append({
        "date": path.name[:10], "cached": cached, **stats
    })
    status = "cached" if cached else "reconstructed"
    print(f"[{index:02d}/{len(files)}] {path.name[:10]}: {daily_table.height:,} rows ({status})")

# This is a materialized DataFrame containing every processed day.
df = pl.concat(daily_tables, how="vertical", rechunk=True).sort("timestamp_utc")
del daily_tables
if df["timestamp_utc"].null_count() or df["timestamp_utc"].n_unique() != df.height:
    raise ValueError("Combined data contains missing or duplicate timestamps")
if not df["timestamp_utc"].is_sorted():
    raise ValueError("Combined data is not sorted by timestamp")
if df.null_count().sum_horizontal().item() != 0:
    raise ValueError("Combined data contains missing values")
if df.filter((pl.col("best_bid") >= pl.col("best_ask")) | (pl.col("book_age_ms") < 0)).height:
    raise ValueError("Combined data contains invalid quotes or future messages")

temporary_output = OUTPUT_PATH.with_suffix(".parquet.tmp")
df.write_parquet(temporary_output, compression="zstd")
temporary_output.replace(OUTPUT_PATH)
book_1s = df  # The previous table name now refers to the complete dataset too.
daily_report = pl.DataFrame(daily_diagnostics)

print(f"\nCombined df: {df.height:,} rows x {df.width} columns")
print(f"UTC range: {df['timestamp_utc'][0]} to {df['timestamp_utc'][-1]}")
print(f"DataFrame size: {df.estimated_size('mb'):.1f} MiB")
print(f"Elapsed: {perf_counter() - start:.1f} seconds")
print(f"Quality: {daily_report['stale_samples_skipped'].sum():,} stale samples skipped; "
      f"{daily_report['invalid_samples_skipped'].sum():,} invalid samples skipped")
print(f"Saved: {OUTPUT_PATH.name} ({OUTPUT_PATH.stat().st_size / 1_000_000:.1f} MB)")
print("First 10 rows:")
display(df.head(10))
print("Last 10 rows:")
display(df.tail(10))

[01/30] 2026-09-24: 86,399 rows (cached)
[02/30] 2026-09-25: 86,399 rows (cached)
[03/30] 2026-09-26: 86,399 rows (cached)
[04/30] 2026-09-27: 86,399 rows (cached)


[05/30] 2026-09-28: 86,399 rows (cached)
[06/30] 2026-09-29: 86,399 rows (cached)


[07/30] 2026-09-30: 86,399 rows (cached)
[08/30] 2026-10-01: 86,399 rows (cached)
[09/30] 2026-10-02: 86,399 rows (cached)
[10/30] 2026-10-03: 86,399 rows (cached)


[11/30] 2026-10-04: 86,399 rows (cached)


[12/30] 2026-10-05: 86,399 rows (cached)
[13/30] 2026-10-06: 86,399 rows (cached)
[14/30] 2026-10-07: 86,399 rows (cached)
[15/30] 2026-10-08: 86,399 rows (cached)


...

[30/30] 2026-10-23: 86,399 rows (cached)



Combined df: 2,591,950 rows x 8 columns
UTC range: 2026-09-24 00:00:01+00:00 to 2026-10-23 23:59:59+00:00
DataFrame size: 158.2 MiB
Elapsed: 1.8 seconds
Quality: 20 stale samples skipped; 0 invalid samples skipped
Saved: btcusdt_lob_1s.parquet (29.0 MB)
First 10 rows:


timestamp_utc,best_bid,bid_qty,best_ask,ask_qty,mid_price,spread,book_age_ms
"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,i64
2026-09-24 00:00:01 UTC,84355.2,1.88,84355.3,3.068,84355.25,0.1,72
2026-09-24 00:00:02 UTC,84355.2,1.295,84355.3,3.781,84355.25,0.1,72
2026-09-24 00:00:03 UTC,84355.2,2.859,84355.3,2.221,84355.25,0.1,71
2026-09-24 00:00:04 UTC,84355.2,2.797,84355.3,2.356,84355.25,0.1,71
2026-09-24 00:00:05 UTC,84355.2,3.354,84355.3,1.664,84355.25,0.1,71
2026-09-24 00:00:06 UTC,84355.2,3.862,84355.3,1.445,84355.25,0.1,71
2026-09-24 00:00:07 UTC,84355.2,3.638,84355.3,1.44,84355.25,0.1,71
2026-09-24 00:00:08 UTC,84355.2,3.861,84355.3,1.44,84355.25,0.1,71
2026-09-24 00:00:09 UTC,84355.2,1.585,84355.3,3.984,84355.25,0.1,71


Last 10 rows:


timestamp_utc,best_bid,bid_qty,best_ask,ask_qty,mid_price,spread,book_age_ms
"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,i64
2026-10-23 23:59:50 UTC,81149.9,1.035,81150.0,1.531,81149.95,0.1,5
2026-10-23 23:59:51 UTC,81149.9,1.059,81150.0,1.531,81149.95,0.1,6
2026-10-23 23:59:52 UTC,81149.9,0.976,81150.0,1.531,81149.95,0.1,5
2026-10-23 23:59:53 UTC,81149.9,0.745,81150.0,1.533,81149.95,0.1,5
2026-10-23 23:59:54 UTC,81149.9,0.745,81150.0,1.41,81149.95,0.1,6
2026-10-23 23:59:55 UTC,81149.9,0.746,81150.0,1.41,81149.95,0.1,5
2026-10-23 23:59:56 UTC,81149.9,0.749,81150.0,1.408,81149.95,0.1,5
2026-10-23 23:59:57 UTC,81149.9,1.586,81150.0,1.057,81149.95,0.1,6
2026-10-23 23:59:58 UTC,81156.6,1.891,81156.7,2.923,81156.65,0.1,5


`df` is the complete reconstructed dataset, sorted by `timestamp_utc`.
`book_1s` is an alias for the same DataFrame. `daily_report` contains
per-file counts and quality diagnostics.

| Column | Meaning |
| --- | --- |
| `timestamp_utc` | Whole-second sampling time in UTC |
| `best_bid`, `bid_qty` | Highest buy price and total BTC quantity there |
| `best_ask`, `ask_qty` | Lowest sell price and total BTC quantity there |
| `mid_price` | `(best_bid + best_ask) / 2`, in USDT |
| `spread` | `best_ask - best_bid`, in USDT |
| `book_age_ms` | Milliseconds since the last message used for that row |

The combined file is `processed/btcusdt_lob_1s.parquet` beside this
notebook. Daily caches are reused only when the raw file, sampling
settings, and reconstruction code match. Initial midnight gaps remain
unfilled rather than inventing observations.

To load the saved DataFrame in a later session, run the cell below.

In [4]:
# Standalone loader: works from HFT/, LOB/, or project subfolders.
from pathlib import Path
import polars as pl
from IPython.display import display

filename = "btcusdt_lob_1s.parquet"
candidates = [
    base / relative / filename
    for base in (Path.cwd(), *Path.cwd().parents)
    for relative in (Path("processed"), Path("LOB") / "processed")
]
OUTPUT_PATH = next((path.resolve() for path in candidates if path.is_file()), None)
if OUTPUT_PATH is None:
    raise FileNotFoundError(
        f"Cannot find {filename} from {Path.cwd()}. Open the HFT project or LOB folder."
    )

df = pl.read_parquet(OUTPUT_PATH)
book_1s = df
print(f"Loaded: {OUTPUT_PATH}")
print(f"df: {df.height:,} rows x {df.width} columns")
display(df.head(5))


Loaded: /mnt/c/Personal_projects/HFT/LOB/processed/btcusdt_lob_1s.parquet
df: 2,591,950 rows x 8 columns


timestamp_utc,best_bid,bid_qty,best_ask,ask_qty,mid_price,spread,book_age_ms
"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,i64
2026-09-24 00:00:01 UTC,84355.2,1.88,84355.3,3.068,84355.25,0.1,72
2026-09-24 00:00:02 UTC,84355.2,1.295,84355.3,3.781,84355.25,0.1,72
2026-09-24 00:00:03 UTC,84355.2,2.859,84355.3,2.221,84355.25,0.1,71
2026-09-24 00:00:04 UTC,84355.2,2.797,84355.3,2.356,84355.25,0.1,71
2026-09-24 00:00:05 UTC,84355.2,3.354,84355.3,1.664,84355.25,0.1,71


In [5]:
# Standalone loader: works from HFT/, LOB/, or project subfolders.
from pathlib import Path
import polars as pl
from IPython.display import display

filename = "btcusdt_lob_1s.parquet"
candidates = [
    base / relative / filename
    for base in (Path.cwd(), *Path.cwd().parents)
    for relative in (Path("processed"), Path("LOB") / "processed")
]
OUTPUT_PATH = next((path.resolve() for path in candidates if path.is_file()), None)
if OUTPUT_PATH is None:
    raise FileNotFoundError(
        f"Cannot find {filename} from {Path.cwd()}. Open the HFT project or LOB folder."
    )

df = pl.read_parquet(OUTPUT_PATH)
book_1s = df
print(f"Loaded: {OUTPUT_PATH}")
print(f"df: {df.height:,} rows x {df.width} columns")
display(df.head(5))


Loaded: /mnt/c/Personal_projects/HFT/LOB/processed/btcusdt_lob_1s.parquet
df: 2,591,950 rows x 8 columns


timestamp_utc,best_bid,bid_qty,best_ask,ask_qty,mid_price,spread,book_age_ms
"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,i64
2026-09-24 00:00:01 UTC,84355.2,1.88,84355.3,3.068,84355.25,0.1,72
2026-09-24 00:00:02 UTC,84355.2,1.295,84355.3,3.781,84355.25,0.1,72
2026-09-24 00:00:03 UTC,84355.2,2.859,84355.3,2.221,84355.25,0.1,71
2026-09-24 00:00:04 UTC,84355.2,2.797,84355.3,2.356,84355.25,0.1,71
2026-09-24 00:00:05 UTC,84355.2,3.354,84355.3,1.664,84355.25,0.1,71


In [8]:
import polars as pl
df = pl.read_parquet("processed/btcusdt_lob_1s.parquet")

In [9]:
df

timestamp_utc,best_bid,bid_qty,best_ask,ask_qty,mid_price,spread,book_age_ms
"datetime[μs, UTC]",f64,f64,f64,f64,f64,f64,i64
2026-09-24 00:00:01 UTC,84355.2,1.88,84355.3,3.068,84355.25,0.1,72
2026-09-24 00:00:02 UTC,84355.2,1.295,84355.3,3.781,84355.25,0.1,72
2026-09-24 00:00:03 UTC,84355.2,2.859,84355.3,2.221,84355.25,0.1,71
2026-09-24 00:00:04 UTC,84355.2,2.797,84355.3,2.356,84355.25,0.1,71
2026-09-24 00:00:05 UTC,84355.2,3.354,84355.3,1.664,84355.25,0.1,71
…,…,…,…,…,…,…,…
2026-10-23 23:59:55 UTC,81149.9,0.746,81150.0,1.41,81149.95,0.1,5
2026-10-23 23:59:56 UTC,81149.9,0.749,81150.0,1.408,81149.95,0.1,5
2026-10-23 23:59:57 UTC,81149.9,1.586,81150.0,1.057,81149.95,0.1,6
